# 3. GRPO
Reinforcement learning with the execution reward, starting from the SFT model. **A100 recommended** (Colab Pro): vLLM generates the groups of replies on the same GPU as training.

1. Keep the prompts the SFT model solves sometimes but not always: they are the only ones GRPO learns from.
2. Train, tracked in Weights & Biases.
3. Evaluate on dev, then once on test, at the very end.

### Setup
The code is cloned from GitHub (`REPO_URL`); set it to `""` to unzip `sql-agent-rl.zip` from your Drive instead. Models and runs are stored in Drive, so they survive the end of the Colab session.

In [ ]:
REPO_URL = "https://github.com/LacosteEvan/sql-agent-rl.git"  # or "" to unzip MyDrive/sql-agent-rl.zip instead
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/sql-agent-rl"
if not os.path.exists("/content/sql-agent-rl"):
    if REPO_URL:
        !git clone -q $REPO_URL /content/sql-agent-rl
    else:
        !unzip -q /content/drive/MyDrive/sql-agent-rl.zip -d /content
%cd /content/sql-agent-rl
!pip install -q -e . -r train/requirements.txt
!python scripts/download_spider.py && python scripts/download_bird.py
!mkdir -p $DRIVE/outputs $DRIVE/runs && ln -sfn $DRIVE/outputs outputs && ln -sfn $DRIVE/runs runs

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "half"  # the free T4 has no bf16

In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
NAME = MODEL.replace("/", "_")
!python -m sql_agent build-sft --split train --out data/sft.jsonl
!wandb login

In [ ]:
!python train/filter_prompts.py --model outputs/sft/merged --data data/sft.jsonl --out data/grpo.jsonl --k 8 --limit 4000

In [ ]:
!python train/grpo.py --model outputs/sft/merged --data data/grpo.jsonl --out outputs/grpo --max-steps 400 --wandb
!python train/merge.py --adapter outputs/grpo --out outputs/grpo/merged

In [ ]:
import subprocess, time, urllib.request

def serve(model, port=8000):
    """Start an OpenAI-compatible vLLM server and wait until it answers."""
    log = open("vllm.log", "w")
    proc = subprocess.Popen(["vllm", "serve", model, "--port", str(port), "--dtype", DTYPE,
                             "--max-model-len", "8192", "--gpu-memory-utilization", "0.9"], stdout=log, stderr=log)
    for _ in range(180):
        if proc.poll() is not None:  # crashed: no point waiting
            break
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models")
            print("vLLM ready:", model)
            return proc
        except Exception:
            time.sleep(5)
    proc.terminate()
    print("".join(open("vllm.log").readlines()[-40:]))
    raise RuntimeError("vLLM did not start: last lines of vllm.log above")

In [ ]:
server = serve("outputs/grpo/merged")
!python -m sql_agent eval --base-url http://localhost:8000/v1 --split dev --workers 32 --model outputs/grpo/merged --max-turns 4 --out runs/{NAME}_grpo_t4.jsonl
!python -m sql_agent eval --base-url http://localhost:8000/v1 --split dev --workers 32 --dataset bird --model outputs/grpo/merged --max-turns 4 --out runs/{NAME}_grpo_t4_bird.jsonl
server.terminate()

In [ ]:
!python -m sql_agent compare runs/{NAME}_sft_t4.jsonl runs/{NAME}_grpo_t4.jsonl
!python -m sql_agent compare runs/{NAME}_base_t4.jsonl runs/{NAME}_grpo_t4.jsonl

### Final test
Run only once the settings are frozen on dev: the test set is for the README table, not for picking hyperparameters. Run it for each model (base, SFT, GRPO), restarting the server on the right model each time.

In [ ]:
server = serve("outputs/grpo/merged")
!python -m sql_agent eval --base-url http://localhost:8000/v1 --split test --workers 32 --model outputs/grpo/merged --max-turns 4 --out runs/{NAME}_grpo_t4_test.jsonl
server.terminate()